# Cube 高级用法

两个进阶能力：Phase 机制（硬件同步降开销）与 MX 低比特矩阵乘。


## 1. Phase：硬件 unit_flag 同步

常规 Cube 计算由框架自动插软件同步。**K 维分块累加后统一搬出、Matmul 结果直送 UB 供 Vector 后处理**（如 FlashAttention 的 Softmax）等场景，软件同步开销可观——Phase 用硬件 `unit_flag` 握手替代：

- `matmul` 系列配 **AccPhase**（Partial/Final）：控制 Cube 流水写 L0C 的硬件握手；
- `store/store_tile/move/insert` 配 **STPhase**：控制 Fixpipe 搬出与 M 流水的握手；
- 未配置时框架自动插软件同步——**无精细控制需求就不配 phase**（仍需正确的 TileGroup+auto_mutex 或手动同步）。

典型场景：K 链（多次累加至同一 L0C 后一次搬出）；Matmul→UB→Vector 后处理链。第 10 章可在正确基线上再评估这一机制。同一 K 链不能混用 Phase 与普通同步：首块覆盖、后续累加，末次 Final 写必须由匹配的 STPhase.Final 搬出消费，不能留下未消费的 flag。


## 2. MX 低比特矩阵乘

`matmul_mx / matmul_mx_acc`：除 L0A/L0B 外，还需 L0A_MX/L0B_MX 中的 **E8M0 量化系数 Tile**（每个系数对应 K 方向连续 32 个尾数元素，**K 必须为 64 的倍数**）。接口支持 MXFP8 的 FP8E4M3FN/E5M2 与 MXFP4 的 FP4E2M1/E1M2；当前 PyPTO Pro 的 `torch.float4_e2m1fn_x2` 普通 Tensor 入参路径对打包后维度的编译绑定与启动校验不一致，FP4 输入可改用 `pl.Ptr[pl.DT_FP4E2M1]` 并在 Kernel 内通过 `pl.make_tensor` 建立逻辑形状视图。从 GM 加载系数时，当前实现要求系数 Tensor 含物理 phase 轴；仅用二维系数 Tensor 会在代码生成阶段报错。完整约束见 matmul_mx 文档——本课程不深入量化细节，只需知道"低比特矩阵乘有专用通路与系数 Tile"。


### K=384 时怎样检查 Phase 的最后一次握手

沿第 4 章的三块 K 累加，前两块是 Partial，最后一块是 Final；最后一次 store 应消费匹配的 Final。首块仍使用 matmul 初始化，后续使用 matmul_acc，Phase 不改变数学累加语义。错误地把每一块都标成 Final，却仅在末尾消费一次，会破坏握手协议。

把 Phase 作为独立实验：先保留无 Phase 的正确基线，再只改这一条 K 链并做相同输入回归，最后测量。MX 的系数 Tile、K 对齐与输入编码则是另一项改造；不能为了同时试多个优化，一次改变 dtype、块大小、Phase 和同步。


## 3. 课后练习

1. （选择）"K 分块累加完成后一次性搬出，软件同步成为热点"——适合启用：
   A. sanitizer<br/>B. Phase（AccPhase + STPhase）<br/>C. 自动流水
2. （填空）MX 矩阵乘中每个 E8M0 系数覆盖 K 方向 ______ 个元素，K 必须是 ______ 的倍数。

In [ ]:
!cat ./answer/08.06_answer.txt

## 4. 本节小结

- Phase = 硬件 unit_flag 替代软件同步，服务 K 链与 M→FIX 精细流水；不配 Phase 时仍须完整的软件同步；
- MX = 低比特矩阵乘专用通路（量化系数 Tile + K 对齐约束）。